# מפגש 26 — פרספטרון: מעבדת OR מודרכת־עצמאית
**90 דקות שיעור +45 דקות פרויקט**. השלימו את `???`. המורה הדגים AND, ואילו אתם עובדים על OR: ארבע דוגמאות [00,01,10,11] ותוויות [0,1,1,1]. אין להסתכל בפתרון המורה. סף המפגש: z>=0 מניב 1.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
X=np.array([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y=np.array([0,1,1,1])
assert X.shape==(4,2)

## 1. חישוב החלטות ידני
נתון w=[1,1], b=−0.5. הציבו את ארבעת זוגות הקלט ב־z=w·x+b. האם z הוא הסתברות?

In [ ]:
fixed_w=np.array([1.,1.])
fixed_b=???
z_values=X @ fixed_w + fixed_b
fixed_preds=np.where(z_values>=???,1,0)
assert np.array_equal(fixed_preds,y)
print(z_values,fixed_preds)

## 2. שינוי משקולות בעקבות טעות
התחילו ב־w=[0,0], b=0, η=0.5; עבור הדוגמה הראשונה OR=[0,0] תתקבל תחזית 1 משום ש־z=0. מה השגיאה ומה ערך ה־bias החדש? שימו לב: במקרה זה המשקולות אינן משתנות כי x=[0,0].

In [ ]:
w0=np.zeros(2)
b0=0.
eta=???
first_pred=1 if np.dot(w0,X[0])+b0>=??? else 0
first_error=y[0]-???
first_w=w0+eta*first_error*X[0]
first_b=b0+eta*first_error
assert first_error==???
assert np.allclose(first_w,[0,0])
assert np.isclose(first_b,-.5)

## 3. פונקציית חיזוי — צעד ראשון לתרגול
בנו פעולת predict שמחזירה את כל התוויות, בסדר הנתונים. אין להשתמש ב־scikit-learn לאימון הפרספטרון בשלב זה.

In [ ]:
def predict_perceptron(X,w,b):
    predictions=[]
    for row in X:
        z=float(np.dot(???,row)+???)
        predicted=1 if z>=??? else 0
        predictions.append(???)
    return np.array(predictions)
assert np.array_equal(predict_perceptron(X,np.array([1.,1.]),-.5),y)

## 4. אימון על OR — תיעוד כל עדכון
לכל דוגמה ב־Train מחשבים חיזוי, טעות, משקולות חדשות ו־bias חדש. אימון נעצר אם אין טעויות בסבב, אך הצלחה על 4 דוגמאות האימון אינה ביצועי Test.

In [ ]:
def train_perceptron(X,y,eta=.5,max_epochs=40):
    w=np.zeros(X.shape[1],dtype=float)
    b=0.
    trace=[]
    for epoch in range(max_epochs):
        mistakes=0
        for i in range(len(X)):
            z=float(np.dot(w,X[i])+b)
            pred=1 if z>=??? else 0
            error=int(y[i])-???
            old=w.copy()
            old_b=b
            if error!=0:
                w=w+???*error*X[i]
                b=b+???*error
                mistakes+=1
            trace.append([epoch,i,z,pred,int(y[i]),error,old[0],old[1],old_b,w[0],w[1],b])
        if mistakes==???:
            break
    names=["epoch","row","z","pred","actual","error","w1_before","w2_before","b_before","w1_after","w2_after","b_after"]
    return w,b,pd.DataFrame(trace,columns=names)
w,b,trace=train_perceptron(X,y)
predictions=predict_perceptron(X,w,b)
assert np.array_equal(predictions,y)
assert len(trace)>0
display(trace.head(12))

## 5. גבול החלטה ו־XOR
הציגו גבול החלטה שהמודל למד ל־OR. מדוע XOR=[0,1,1,0] לא ניתן להפרדה בישר אחד? אין צורך להציג רשת עמוקה או להעתיק פתרון עתידי.

In [ ]:
plt.figure()
plt.scatter(X[:,0],X[:,1],c=y,cmap="coolwarm",vmin=0,vmax=1)
grid=np.array([-.2,1.2])
if abs(w[1])>1e-12:
    plt.plot(grid,-(w[0]*grid+b)/w[1],color="black")
elif abs(w[0])>1e-12:
    plt.axvline(-b/w[0],color="black")
plt.xlim(-.2,1.2);plt.ylim(-.2,1.2)
plt.xlabel("x1");plt.ylabel("x2");plt.title("OR decision boundary")
plt.show()
xor_y=np.array([0,1,1,0])
assert np.array_equal(???,np.array([0,1,1,0]))

## כרטיס יציאה
הסבירו מהו z, מהו הסף, מה עושה η, כיצד מעדכנים b, מה ההבדל בין הצלחת אימון להכללה, ומדוע קו יחיד לא יפתור XOR. התרגול משמש גשר לנושא רשתות בהמשך, בלי ללמד כאן Backpropagation.